In [2]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

#### Loading labels from the pickled data

In [6]:
def load_wesad_data(subject_id):
    file_path = f'WESAD/S{subject_id}/S{subject_id}.pkl'
    with open(file_path, 'rb') as file:
        data = pickle.load(file, encoding='latin1')
    return data

def extract_labels(data):
    labels = data['label']
    return labels

subject_id = 5  # Example subject
data = load_wesad_data(subject_id)
labels = extract_labels(data)

print(np.unique(labels))

[0 1 2 3 4 5 6 7]


#### Experimenting

In [9]:
subject_id = 4
data = load_wesad_data(subject_id)

# # Print keys in the data dictionary
# print(data.keys())

# # Print label information
# print("Labels:")
# print(data['label'])

# # Print signal data keys
# print("Signal keys:")
# print(data['signal'].keys())

# # Print wrist signal keys (Empatica E4)
# print("Wrist signal keys:")
# print(data['signal']['wrist'].keys())

# # Print a snippet of accelerometer data from wrist
print("Wrist ACC data (first few entries):")
print(data['label'][:100])


Wrist ACC data (first few entries):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


#### Preprocessing pipeline

In [90]:
# Load data
def load_empatica_data(subject_id, base_path='WESAD'):
    # base_path = f'WESAD/{subject_id}/{subject_id}_E4_Data/'
    base_path = f'WESAD/{subject_id}/{subject_id}_E4_Data/'
    bvp = pd.read_csv(base_path + 'BVP.csv', header=None, encoding='utf-8')
    eda = pd.read_csv(base_path + 'EDA.csv', header=None, encoding='utf-8')
    hr = pd.read_csv(base_path + 'HR.csv', header=None, encoding='utf-8')
    temp = pd.read_csv(base_path + 'TEMP.csv', header=None, encoding='utf-8')
    return acc, bvp, eda, hr, temp

# Generate timestamps based on the initial time and sampling rate
def generate_timestamps(start_time, sampling_rate, length):
    return start_time + np.arange(length) / sampling_rate

# Process each sensor data
def process_sensor_data(df, column_names):
    start_time = df.iloc[0, 0]  # Initial timestamp
    sampling_rate = df.iloc[1, 0]  # Sampling rate
    df = df.iloc[2:]  # Actual data starts from the third row
    df.columns = column_names
    df.reset_index(drop=True, inplace=True)
    timestamps = generate_timestamps(start_time, sampling_rate, len(df))
    df = df.copy()
    df.loc[:, 'timestamp'] = timestamps
    return df

# Align and preprocess data
def preprocess_data(acc, bvp, eda, hr, temp):
    acc = process_sensor_data(acc, ['X', 'Y', 'Z'])
    bvp = process_sensor_data(bvp, ['BVP'])
    eda = process_sensor_data(eda, ['EDA'])
    hr = process_sensor_data(hr, ['HR'])
    temp = process_sensor_data(temp, ['TEMP'])

    # Resample or interpolate data to have the same timestamps
    min_timestamp = max(acc['timestamp'].min(), bvp['timestamp'].min(), eda['timestamp'].min(), hr['timestamp'].min(), temp['timestamp'].min())
    max_timestamp = min(acc['timestamp'].max(), bvp['timestamp'].max(), eda['timestamp'].max(), hr['timestamp'].max(), temp['timestamp'].max())

    common_timestamps = np.arange(min_timestamp, max_timestamp, 1)  # 1 second intervals, adjust as necessary

    def interpolate(df, timestamps):
        interp_func = interp1d(df['timestamp'], df.iloc[:, :-1], axis=0, fill_value="extrapolate")
        interpolated_data = interp_func(timestamps)
        return pd.DataFrame(interpolated_data, columns=df.columns[:-1], index=timestamps)
    
    acc_interpolated = interpolate(acc, common_timestamps)
    bvp_interpolated = interpolate(bvp, common_timestamps)
    eda_interpolated = interpolate(eda, common_timestamps)
    hr_interpolated = interpolate(hr, common_timestamps)
    temp_interpolated = interpolate(temp, common_timestamps)


    # Inside preprocess_data function after interpolation
    print(f"acc shape after interpolation: {acc_interpolated.shape}")
    print(f"bvp shape after interpolation: {bvp_interpolated.shape}")
    # Print other data shapes as needed

    # Combine all data into a single DataFrame
    combined_data = pd.concat([acc_interpolated, bvp_interpolated, eda_interpolated, hr_interpolated, temp_interpolated], axis=1)

    # Handle missing data (if any)
    combined_data.ffill(inplace=True)

    # Inside prepare_data function after aligning labels
    print(f"combined_data shape: {combined_data.shape}")

    # # Normalize or standardize data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(combined_data)
    return pd.DataFrame(scaled_data, columns=combined_data.columns, index=common_timestamps)


In [91]:
# Load and attach labels
def load_labels(subject_id, base_path='WESAD'):
    with open(f'{base_path}/{subject_id}/{subject_id}.pkl', 'rb') as file:
        data = pickle.load(file, encoding="latin1")
    labels = data['label']
    return labels

# Main function to load, preprocess, and attach labels
def prepare_data(subject_ids, base_path='WESAD'):
    preprocessed_data_dict = {}
    all_data = []
    all_labels = []

    for subject_id in subject_ids:
        acc, bvp, eda, hr, temp = load_empatica_data(subject_id, base_path)
        preprocessed_data = preprocess_data(acc, bvp, eda, hr, temp)
        labels = load_labels(subject_id, base_path)
        
        # Align labels with preprocessed data timestamps
        min_timestamp = preprocessed_data.index.min()
        max_timestamp = preprocessed_data.index.max()
        aligned_labels = labels[(labels.index >= min_timestamp) & (labels.index <= max_timestamp)]
        aligned_labels = aligned_labels.reindex(preprocessed_data.index, method='nearest')
        print(aligned_labels.shape)  # Check the shape of aligned_labels
        # Append to combined data
        preprocessed_data['label'] = aligned_labels
        preprocessed_data = preprocessed_data[preprocessed_data['label'].isin([0, 1, 2, 3, 4])]  # Exclude unwanted labels
        all_data.append(preprocessed_data.drop(columns=['label']))
        all_labels.extend(preprocessed_data['label'])
        
        preprocessed_data_dict[subject_id] = preprocessed_data
        print(f"Preprocessed data for {subject_id} completed.")

    combined_data = pd.concat(all_data)
    combined_labels = np.array(all_labels)
 
    print(combined_data.shape)  # Check the shape of combined_data
    print(combined_data.head())  # Print the first few rows to inspect the data
    return combined_data, combined_labels

In [ ]:
# # List of subject IDs in WESAD dataset (assuming 'S1' to 'S15')
# subject_ids = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

# # Dictionary to store preprocessed data for each subject
# preprocessed_data_dict = {}

# # Preprocess data for each subject
# for subject_id in subject_ids:
#     acc, bvp, eda, hr, temp = load_empatica_data(subject_id)
#     preprocessed_data = preprocess_data(acc, bvp, eda, hr, temp)
#     preprocessed_data_dict[subject_id] = preprocessed_data
 
#     # Print status
#     print(f"Preprocessed data for {subject_id} completed.")

In [92]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Subject IDs
subject_ids = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

# Prepare data
X, y = prepare_data(subject_ids)
 
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Machine learning model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred))
print('Accuracy:', accuracy_score(y_test, y_pred))

acc shape after interpolation: (0, 3)
bvp shape after interpolation: (0, 1)
combined_data shape: (0, 7)


ValueError: Found array with 0 sample(s) (shape=(0, 7)) while a minimum of 1 is required by StandardScaler.